In [ ]:
import os
import gc
import datetime
import numpy as np
import pandas as pd
from glob import glob
import dask.dataframe as dd

In [ ]:
code = 'B120'

max_row = 500000
files_folder_path = f"{code}/{code}_output/"
combine_folder_path = f"{code}/{code}_Dashboard/"
dte_file = pd.read_csv(f"C:/PICKLE/DTE.csv", parse_dates=['Date'], dayfirst=True).set_index("Date")
os.makedirs(combine_folder_path, exist_ok=True)

EXT = "*.parquet"
all_files = [file for path, subdir, files in os.walk(files_folder_path) for file in glob(os.path.join(path, EXT))]
indices = set([f.split('\\')[-1].split(' ')[0] for f in all_files])

print('Number of Files -', len(all_files))
print()

print('Indices -', indices)
print()

df = pd.read_parquet(max([(file, os.path.getsize(file)) for file in all_files], key=lambda x: x[-1])[0])
name_columns = [c for c in list(df.columns) if c.startswith('P_')]
pnl_columns = [c for c in list(df.columns) if c.endswith('PNL')]
print('Name cols-', list(map(lambda x: x.replace('P_', ''), name_columns)))
print()
print('PNL cols-', pnl_columns)

year_day_dte_files = {}

for file in all_files:
    index = file.split('\\')[-1].split(' ')[0]
    date = datetime.datetime.strptime(file.split('\\')[-1].split(' ')[1], "%Y-%m-%d")
    year = date.year
    day = date.strftime('%A')
    dte = dte_file.loc[date, index]
    year_day_dte_files[f'{index}-{year}-{day}-{dte}'] = year_day_dte_files.get(f'{index}-{year}-{day}-{dte}', []) + [file]

In [ ]:
print('Building DashBoard Files... \n')
for index in indices:
    os.makedirs(f"{combine_folder_path}{index}", exist_ok=True)
    
    for key, value in year_day_dte_files.items():
        check_index, year, day, dte = key.split('-')
        if check_index != index: continue
            
        dashboard_data_list = []
        chunks = sorted(set([f.split(' ')[-1].split('.')[0] for f in year_day_dte_files[key]]), key=lambda x: int(x.split('-')[-1]))
        for chunk in chunks:
            
            print(index, year, day, dte, chunk)

            chunks_file = [f for f in year_day_dte_files[key] if f"{chunk}." in f]
            data = dd.read_parquet(chunks_file, columns=(name_columns+pnl_columns))
            data = data.compute()
            data = data.groupby(name_columns).sum(numeric_only=True)[pnl_columns].reset_index()
            data = data.melt(id_vars=name_columns, value_vars=pnl_columns, var_name='PL Basis', value_name='Points')
            data.columns = [c.replace('P_','') for c in data.columns]
            data[['Year', 'Day', 'DTE']] = np.int16(year), day, np.int8(float(dte))
            dashboard_data_list.append(data)

        dashboard_data = pd.concat(dashboard_data_list, ignore_index=True)
        dashboard_data[dashboard_data.select_dtypes(include=['object']).columns] = dashboard_data.select_dtypes(include=['object']).astype('category')

        for pnl_col in pnl_columns:
            pnl_data = dashboard_data[dashboard_data['PL Basis'] == pnl_col]
            chunk_size = max_row
            for idx, i in enumerate(range(0, len(pnl_data), chunk_size), start=1):
                chunk = pnl_data.iloc[i:i + chunk_size]
                chunk.to_csv(f"{combine_folder_path}{index}/{code}-{year}-{day}-{dte}-{pnl_col}-No-{idx}.csv", index=False)
                
        del dashboard_data
        del dashboard_data_list
        gc.collect()